In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    for m_col in [f'M{i}' for i in range(1, 19)]:
        for e_col in [f'E{i}' for i in range(1, 21)]:
            data[f'ME_prod_{m_col}_{e_col}'] = data[m_col] * data[e_col]
            data[f'ME_ratio_{m_col}_{e_col}'] = data[m_col] / (data[e_col] + 1e-8)
            
            main_features.append(f'ME_prod_{m_col}_{e_col}')
            main_features.append(f'ME_ratio_{m_col}_{e_col}')
            
    # M × I (Market × Interest Rate)
    for m_col in [f'M{i}' for i in range(1, 19)]:
        for i_col in [f'I{i}' for i in range(1, 10)]:
            data[f'MI_prod_{m_col}_{i_col}'] = data[m_col] * data[i_col]
            data[f'MI_ratio_{m_col}_{i_col}'] = data[m_col] / (data[i_col] + 1e-8)
            data[f'MI_spread_{m_col}_{i_col}'] = data[m_col] - data[i_col]

            main_features.append(f'MI_prod_{m_col}_{i_col}')
            main_features.append(f'MI_ratio_{m_col}_{i_col}')
            main_features.append(f'MI_spread_{m_col}_{i_col}')
        
    # M × P (Market × Price)
    for m_col in [f'M{i}' for i in range(1, 19)]:
        for p_col in ['P8', 'P9', 'P10', 'P12', 'P13']:
            data[f'MP_prod_{m_col}_{p_col}'] = data[m_col] * data[p_col]
            data[f'MP_ratio_{m_col}_{p_col}'] = data[m_col] / (data[p_col] + 1e-8)

            main_features.append(f'MP_prod_{m_col}_{p_col}')
            main_features.append(f'MP_ratio_{m_col}_{p_col}')
    
    # M × V (Market × Volatility)
    for m_col in [f'M{i}' for i in range(1, 19)]:
        for v_col in [f'V{i}' for i in range(1, 14)]:
            data[f'MV_ratio_{m_col}_{v_col}'] = data[m_col] / (data[v_col] + 1e-8)
            data[f'MV_prod_{m_col}_{v_col}'] = data[m_col] * data[v_col]
            # Sharpe-like ratio
            if 'return' in m_col.lower():
                data[f'MV_sharpe_{m_col}_{v_col}'] = data[m_col] / (data[v_col] + 1e-8)
                main_features.append(f'MV_sharpe_{m_col}_{v_col}')
                
            main_features.append(f'MV_ratio_{m_col}_{v_col}')
            main_features.append(f'MV_prod_{m_col}_{v_col}')
        
    # M × S (Market × Sentiment)
    for m_col in [f'M{i}' for i in range(1, 19)]:
        for s_col in [f'S{i}' for i in range(1, 13)]:
            data[f'MS_prod_{m_col}_{s_col}'] = data[m_col] * data[s_col]
            data[f'MS_weighted_{m_col}_{s_col}'] = data[m_col] * (1 + data[s_col])
            main_features.append(f'MS_prod_{m_col}_{s_col}')
            main_features.append(f'MS_weighted_{m_col}_{s_col}')
        
    # E × I (Economic × Interest Rate)
    for e_col in [f'E{i}' for i in range(1, 21)]:
        for i_col in [f'I{i}' for i in range(1, 10)]:
            data[f'EI_diff_{e_col}_{i_col}'] = data[e_col] - data[i_col]
            data[f'EI_ratio_{e_col}_{i_col}'] = data[e_col] / (data[i_col] + 1e-8)
            data[f'EI_prod_{e_col}_{i_col}'] = data[e_col] * data[i_col]
            main_features.append(f'EI_diff_{e_col}_{i_col}')
            main_features.append(f'EI_ratio_{e_col}_{i_col}')
            main_features.append(f'EI_prod_{e_col}_{i_col}')
        
    # E × P (Economic × Price)
    for e_col in [f'E{i}' for i in range(1, 21)]:
        for p_col in ['P8', 'P9', 'P10', 'P12', 'P13']:
            data[f'EP_prod_{e_col}_{p_col}'] = data[e_col] * data[p_col]
            data[f'EP_ratio_{e_col}_{p_col}'] = data[e_col] / (data[p_col] + 1e-8)
            main_features.append(f'EP_prod_{e_col}_{p_col}')
            main_features.append(f'EP_ratio_{e_col}_{p_col}')
        
    # E × V (Economic × Volatility)
    for e_col in [f'E{i}' for i in range(1, 21)]:
        for v_col in [f'V{i}' for i in range(1, 14)]:
            data[f'EV_prod_{e_col}_{v_col}'] = data[e_col] * data[v_col]
            data[f'EV_uncertainty_{e_col}_{v_col}'] = abs(data[e_col]) * data[v_col]
            main_features.append(f'EV_prod_{e_col}_{v_col}')
            main_features.append(f'EV_uncertainty_{e_col}_{v_col}')
        
    # E × S (Economic × Sentiment)
    for e_col in [f'E{i}' for i in range(1, 21)]:
        for s_col in [f'S{i}' for i in range(1, 13)]:
            data[f'ES_prod_{e_col}_{s_col}'] = data[e_col] * data[s_col]
            data[f'ES_diff_{e_col}_{s_col}'] = data[e_col] - data[s_col]
            main_features.append(f'ES_prod_{e_col}_{s_col}')
            main_features.append(f'ES_diff_{e_col}_{s_col}')
        
    # I × P (Interest Rate × Price)
    for i_col in [f'I{i}' for i in range(1, 10)]:
        for p_col in ['P8', 'P9', 'P10', 'P12', 'P13']:
            data[f'IP_prod_{i_col}_{p_col}'] = data[i_col] * data[p_col]
            data[f'IP_discount_{i_col}_{p_col}'] = data[p_col] / (1 + data[i_col] + 1e-8)
            main_features.append(f'IP_prod_{i_col}_{p_col}')
            main_features.append(f'IP_discount_{i_col}_{p_col}')

    # I × V (Interest Rate × Volatility)
    for i_col in [f'I{i}' for i in range(1, 10)]:
        for v_col in [f'V{i}' for i in range(1, 14)]:
            data[f'IV_prod_{i_col}_{v_col}'] = data[i_col] * data[v_col]
            main_features.append(f'IV_prod_{i_col}_{v_col}')
        
    # I × S (Interest Rate × Sentiment)
    for i_col in [f'I{i}' for i in range(1, 10)]:
        for s_col in [f'S{i}' for i in range(1, 13)]:
            data[f'IS_prod_{i_col}_{s_col}'] = data[i_col] * data[s_col]
            main_features.append(f'IS_prod_{i_col}_{s_col}')
        
    # P × V (Price × Volatility)
    for p_col in ['P8', 'P9', 'P10', 'P12', 'P13']:
        for v_col in [f'V{i}' for i in range(1, 14)]:
            data[f'PV_prod_{p_col}_{v_col}'] = data[p_col] * data[v_col]
            data[f'PV_stability_{p_col}_{v_col}'] = data[p_col] / (data[v_col] + 1e-8)
            main_features.append(f'PV_prod_{p_col}_{v_col}')
            main_features.append(f'PV_stability_{p_col}_{v_col}')
        
    # P × S (Price × Sentiment)
    for p_col in ['P8', 'P9', 'P10', 'P12', 'P13']:
        for s_col in [f'S{i}' for i in range(1, 13)]:
            data[f'PS_prod_{p_col}_{s_col}'] = data[p_col] * data[s_col]
            # Contrarian signal (opposite signs amplify)
            data[f'PS_contrarian_{p_col}_{s_col}'] = -data[p_col] * data[s_col]
            main_features.append(f'PS_prod_{p_col}_{s_col}')
            main_features.append(f'PS_contrarian_{p_col}_{s_col}')
        
    # V × S (Volatility × Sentiment)
    for v_col in [f'V{i}' for i in range(1, 14)]:
        for s_col in [f'S{i}' for i in range(1, 13)]:
            data[f'VS_prod_{v_col}_{s_col}'] = data[v_col] * data[s_col]
            data[f'VS_panic_{v_col}_{s_col}'] = data[v_col] * abs(data[s_col])
            main_features.append(f'VS_prod_{v_col}_{s_col}')
            main_features.append(f'VS_panic_{v_col}_{s_col}')
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,VS_panic_V13_S8,VS_prod_V13_S9,VS_panic_V13_S9,VS_prod_V13_S10,VS_panic_V13_S10,VS_prod_V13_S11,VS_panic_V13_S11,VS_prod_V13_S12,VS_panic_V13_S12,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,-0.110812,-0.616679,-0.616679,-0.529658,-0.529658,-0.283237,-0.283237,0.022213,-0.022213,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,-0.098375,-0.486653,-0.486653,-0.334998,-0.334998,-0.259548,-0.259548,-0.448902,-0.448902,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,-0.116045,-0.594271,-0.594271,-0.366865,-0.366865,-0.251696,-0.251696,-0.562051,-0.562051,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,-0.089659,-0.632680,-0.632680,-0.102609,-0.102609,-0.063363,-0.063363,-1.711014,-1.711014,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,-0.161997,-0.449950,-0.449950,-0.698119,-0.698119,-0.551600,-0.551600,-0.135051,-0.135051,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,-0.248416,-0.323640,-0.323640,-0.118077,-0.118077,-0.058324,-0.058324,-0.274268,-0.274268,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,-0.249514,-0.159076,-0.159076,-0.400815,-0.400815,-0.309915,-0.309915,-0.520280,-0.520280,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,-0.248281,-0.203164,-0.203164,-0.337668,-0.337668,-0.293490,-0.293490,-0.376610,-0.376610,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,-0.242692,-0.352563,-0.352563,-0.004585,-0.004585,-0.010190,-0.010190,0.000688,-0.000688,0.008310


In [4]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBM_R_model = joblib.load('LGBM_R.pkl')

LGBMRegressor TRAINING...


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.7181617586697819


In [6]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))